## Week 6 - Spark Architecture and Performance

**Objective:**
Understand Spark architecture and perform efficient data processing using transformations, filtering, schema handling, and optimized file formats (CSV and Parquet).

**Dataset:** Ecommerce Sales Dataset

In [ ]:
#Importing Libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
#Created SparkSession
spark = SparkSession.builder \
    .appName("Week6 Spark Architecture") \
    .getOrCreate()

In [ ]:
#Verifying SparkSession
spark

In [ ]:
#Reading the CSV
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../data/ecommerce_sales_34500.csv")

In [ ]:
#Showing First 5 Rows
df.show(5)

+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+
|order_id|customer_id|product_id|   category| price|discount|quantity|payment_method|order_date|delivery_time_days|region|returned|total_amount|shipping_cost|profit_margin|customer_age|customer_gender|
+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+
| O100000|     C17270|   P234890|       Home|164.08|    0.15|       1|   Credit Card|2023-12-23|                 4|  West|      No|      139.47|         7.88|        31.17|          60|         Female|
| O100001|     C17603|   P228204|    Grocery| 24.73|     0.0|       1|   Credit Card|2025-04-03|                 6| South|      No|       24.73|          4.6|        -2.62|          37|       

In [ ]:
#Counting Records ensuring the dataset loaded succesfully
print("Total Records:", df.count())

Total Records: 34500


# Now exploring the dataset

In [ ]:
#printing column names
print(df.columns)

['order_id', 'customer_id', 'product_id', 'category', 'price', 'discount', 'quantity', 'payment_method', 'order_date', 'delivery_time_days', 'region', 'returned', 'total_amount', 'shipping_cost', 'profit_margin', 'customer_age', 'customer_gender']


In [10]:
#printing dataset information
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_time_days: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- returned: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- profit_margin: double (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- customer_gender: string (nullable = true)



In [11]:
df.show(10, truncate=False)

+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+
|order_id|customer_id|product_id|category   |price |discount|quantity|payment_method|order_date|delivery_time_days|region|returned|total_amount|shipping_cost|profit_margin|customer_age|customer_gender|
+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+
|O100000 |C17270     |P234890   |Home       |164.08|0.15    |1       |Credit Card   |2023-12-23|4                 |West  |No      |139.47      |7.88         |31.17        |60          |Female         |
|O100001 |C17603     |P228204   |Grocery    |24.73 |0.0     |1       |Credit Card   |2025-04-03|6                 |South |No      |24.73       |4.6          |-2.62        |37          |Male   

In [12]:
print("Total Columns:", len(df.columns))

Total Columns: 17


In [ ]:
#checking for null values
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

+--------+-----------+----------+--------+-----+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+
|order_id|customer_id|product_id|category|price|discount|quantity|payment_method|order_date|delivery_time_days|region|returned|total_amount|shipping_cost|profit_margin|customer_age|customer_gender|
+--------+-----------+----------+--------+-----+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+
|       0|          0|         0|       0|    0|       0|       0|             0|         0|                 0|     0|       0|           0|            0|            0|           0|              0|
+--------+-----------+----------+--------+-----+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+



In [14]:
electronics_df = df.filter(
    col("category") == "Electronics"
).select(
    "product_id",
    "price"
)

electronics_df.show(10)

+----------+------+
|product_id| price|
+----------+------+
|   P213892|175.58|
|   P208689| 63.67|
|   P201363| 266.5|
|   P218405|734.32|
|   P203302| 128.8|
|   P243790|313.25|
|   P229316|402.13|
|   P223089|368.67|
|   P207676| 97.95|
|   P212663| 753.1|
+----------+------+
only showing top 10 rows


In [ ]:
#filtering orders above Rs 1000
high_value_orders = df.filter(
    col("total_amount") > 1000
)

high_value_orders.show(10)

+--------+-----------+----------+-----------+-------+--------+--------+--------------+----------+------------------+-------+--------+------------+-------------+-------------+------------+---------------+
|order_id|customer_id|product_id|   category|  price|discount|quantity|payment_method|order_date|delivery_time_days| region|returned|total_amount|shipping_cost|profit_margin|customer_age|customer_gender|
+--------+-----------+----------+-----------+-------+--------+--------+--------------+----------+------------------+-------+--------+------------+-------------+-------------+------------+---------------+
| O100018|     C17831|   P218405|Electronics| 734.32|     0.0|       2|        Wallet|2025-06-20|                 4|  South|      No|     1468.64|         10.9|       165.34|          58|         Female|
| O100092|     C13073|   P230373|Electronics| 857.18|     0.0|       3|           COD|2024-08-10|                 4|   West|      No|     2571.54|        12.41|       296.17|          

In [16]:
north_orders = df.filter(
    col("region") == "North"
)

north_orders.show(10)

+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+
|order_id|customer_id|product_id|   category| price|discount|quantity|payment_method|order_date|delivery_time_days|region|returned|total_amount|shipping_cost|profit_margin|customer_age|customer_gender|
+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+---------------+
| O100002|     C10860|   P213892|Electronics|175.58|    0.05|       1|   Credit Card|2024-10-08|                 4| North|      No|       166.8|         6.58|        13.44|          34|           Male|
| O100013|     C15578|   P205339|       Toys|  7.12|     0.0|       1|    Debit Card|2023-11-01|                 5| North|      No|        7.12|         3.35|         -0.5|          56|       

## Data Transformation

In this section, we perform common Spark DataFrame transformations such as:

- Renaming columns
- Casting data types
- Creating new columns
- Filtering transformed data

In [17]:
df = df.withColumnRenamed("customer_gender", "gender")

df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_time_days: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- returned: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- profit_margin: double (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- gender: string (nullable = true)



In [18]:
#explicitly casting data type of price which is already double
#but to show that how to do type casting
from pyspark.sql.functions import col

df = df.withColumn(
    "price",
    col("price").cast("double")
)

df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_time_days: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- returned: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- profit_margin: double (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- gender: string (nullable = true)



In [19]:
#creating required column using existing col
df = df.withColumn(
    "final_price",
    col("price") * 1.18
)

df.select(
    "product_id",
    "price",
    "final_price"
).show(10)

+----------+------+------------------+
|product_id| price|       final_price|
+----------+------+------------------+
|   P234890|164.08|193.61440000000002|
|   P228204| 24.73|           29.1814|
|   P213892|175.58|          207.1844|
|   P208689| 63.67|           75.1306|
|   P228063| 16.33|19.269399999999997|
|   P214062| 53.91| 63.61379999999999|
|   P201363| 266.5|314.46999999999997|
|   P216691|  9.98|           11.7764|
|   P202751|  6.61|            7.7998|
|   P207782| 10.91|           12.8738|
+----------+------+------------------+
only showing top 10 rows


In [20]:
#creating status column
df = df.withColumn(
    "status",
    when(
        col("returned") == "No",
        "Completed"
    ).otherwise("Returned")
)

df.select(
    "returned",
    "status"
).show(10)

+--------+---------+
|returned|   status|
+--------+---------+
|      No|Completed|
|      No|Completed|
|      No|Completed|
|      No|Completed|
|      No|Completed|
|      No|Completed|
|      No|Completed|
|      No|Completed|
|      No|Completed|
|      No|Completed|
+--------+---------+
only showing top 10 rows


In [21]:
#creating priority column
df = df.withColumn(
    "priority",
    when(
        col("total_amount") > 3000,
        "High"
    ).otherwise("Low")
)

df.select(
    "total_amount",
    "priority"
).show(10)

+------------+--------+
|total_amount|priority|
+------------+--------+
|      139.47|     Low|
|       24.73|     Low|
|       166.8|     Low|
|       63.67|     Low|
|       13.88|     Low|
|       97.04|     Low|
|       266.5|     Low|
|        9.98|     Low|
|        6.28|     Low|
|       10.91|     Low|
+------------+--------+
only showing top 10 rows


In [22]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_time_days: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- returned: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- profit_margin: double (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- final_price: double (nullable = true)
 |-- status: string (nullable = false)
 |-- priority: string (nullable = false)



# Why are we creating status and priority?

Because the assignment includes these columns, but my dataset doesn't.

Instead of downloading another dataset,i created and edited existing columns as per assignment requirment

## Filtering Data

Applying different filtering conditions using Spark DataFrames.

In [23]:
#since we have status column now
completed_orders = df.filter(
    (col("status") == "Completed") &
    (col("total_amount") > 1000)
)

completed_orders.show(10)

+--------+-----------+----------+-----------+-------+--------+--------+--------------+----------+------------------+-------+--------+------------+-------------+-------------+------------+------+------------------+---------+--------+
|order_id|customer_id|product_id|   category|  price|discount|quantity|payment_method|order_date|delivery_time_days| region|returned|total_amount|shipping_cost|profit_margin|customer_age|gender|       final_price|   status|priority|
+--------+-----------+----------+-----------+-------+--------+--------+--------------+----------+------------------+-------+--------+------------+-------------+-------------+------------+------+------------------+---------+--------+
| O100018|     C17831|   P218405|Electronics| 734.32|     0.0|       2|        Wallet|2025-06-20|                 4|  South|      No|     1468.64|         10.9|       165.34|          58|Female|          866.4976|Completed|     Low|
| O100092|     C13073|   P230373|Electronics| 857.18|     0.0|      

In [24]:
#using or condition
north_or_high = df.filter(
    (col("region") == "North") |
    (col("priority") == "High")
)

north_or_high.show(10)

+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+------+------------------+---------+--------+
|order_id|customer_id|product_id|   category| price|discount|quantity|payment_method|order_date|delivery_time_days|region|returned|total_amount|shipping_cost|profit_margin|customer_age|gender|       final_price|   status|priority|
+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+------+------------------+---------+--------+
| O100002|     C10860|   P213892|Electronics|175.58|    0.05|       1|   Credit Card|2024-10-08|                 4| North|      No|       166.8|         6.58|        13.44|          34|  Male|          207.1844|Completed|     Low|
| O100013|     C15578|   P205339|       Toys|  7.12|     0.0|       1|    De

In [ ]:
#handling null values but we dont have any null val in
#the dataset it does no change
clean_df = df.na.drop()

clean_df.show(5)

+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+------+------------------+---------+--------+
|order_id|customer_id|product_id|   category| price|discount|quantity|payment_method|order_date|delivery_time_days|region|returned|total_amount|shipping_cost|profit_margin|customer_age|gender|       final_price|   status|priority|
+--------+-----------+----------+-----------+------+--------+--------+--------------+----------+------------------+------+--------+------------+-------------+-------------+------------+------+------------------+---------+--------+
| O100000|     C17270|   P234890|       Home|164.08|    0.15|       1|   Credit Card|2023-12-23|                 4|  West|      No|      139.47|         7.88|        31.17|          60|Female|193.61440000000002|Completed|     Low|
| O100001|     C17603|   P228204|    Grocery| 24.73|     0.0|       1|   Cre

## Convert CSV to Parquet

In [ ]:
# Spark Parquet Write (Reference)
# This command works in environments where Hadoop/WinUtils is properly configured.

#The following Spark write operation is the 
#standard approach for saving data as Parquet:

# df.write \
#     .mode("overwrite") \
#     .parquet("../data/ecommerce_sales_34500.parquet")

In [29]:



#Saving the Final Dataset as CSV Using Pandas


# Convert Spark DataFrame to Pandas
pandas_df = clean_df.toPandas()

# Save CSV
pandas_df.to_csv(
    "../outputs/csv_output.csv",
    index=False
)

print("CSV file saved successfully.")

CSV file saved successfully.


In [31]:
import pandas as pd

result = pd.read_csv("../outputs/csv_output.csv")

result.head()

,order_id,customer_id,product_id,category,price,discount,quantity,payment_method,order_date,delivery_time_days,region,returned,total_amount,shipping_cost,profit_margin,customer_age,gender,final_price,status,priority
0,O100000,C17270,P234890,Home,164.08,0.15,1,Credit Card,2023-12-23,4,West,No,139.47,7.88,31.17,60,Female,193.6144,Completed,Low
1,O100001,C17603,P228204,Grocery,24.73,0.00,1,Credit Card,2025-04-03,6,South,No,24.73,4.60,-2.62,37,Male,29.1814,Completed,Low
2,O100002,C10860,P213892,Electronics,175.58,0.05,1,Credit Card,2024-10-08,4,North,No,166.80,6.58,13.44,34,Male,207.1844,Completed,Low
3,O100003,C15390,P208689,Electronics,63.67,0.00,1,UPI,2024-09-14,6,South,No,63.67,5.50,2.14,21,Female,75.1306,Completed,Low
4,O100004,C15226,P228063,Home,16.33,0.15,1,COD,2024-12-21,6,East,No,13.88,2.74,1.15,39,Male,19.2694,Completed,Low


In [32]:
print("Rows:", result.shape[0])
print("Columns:", result.shape[1])

Rows: 34500
Columns: 20


In [33]:
print("Week 6 Spark Assignment Completed Successfully!")

Week 6 Spark Assignment Completed Successfully!
